<a href="https://colab.research.google.com/github/liminalvoid/nlp/blob/main/sem_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Семинар 4. Word2Vec и простой retrieval

## Настройка, установка и импорт необходимых библиотек

In [23]:
!pip install -q gensim

import re
import zipfile
import os
import tempfile

import numpy as np
import gensim
import gensim.downloader as api
import requests

from collections import Counter
from typing import List

from datasets import load_dataset
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

## Загрузка модели эмбеддингов для русского языка, анализ требуемого препроцессинга

В качестве модели эмбеддингов для русского языка используется [ruscorpora_upos_cbow_300_20_2019](https://rusvectores.org/en/models/#:~:text=ruscorpora%5Fupos%5Fcbow%5F300%5F20%5F2019). Скачаем архив с моделью.

In [17]:
MODEL_PATH = "./model.zip"
MODEL_URL = "https://vectors.nlpl.eu/repository/20/180.zip"

r = requests.get(MODEL_URL)

with open(MODEL_PATH, "wb") as f:
    f.write(r.content)


Далее загрузим модель из скаченного архива.

In [18]:
model = None

with zipfile.ZipFile(MODEL_PATH, "r") as archive:
    with tempfile.TemporaryDirectory() as tmp:
        archive.extract("model.txt", path=tmp)
        txt_path = os.path.join(tmp, "model.txt")
        model = gensim.models.KeyedVectors.load_word2vec_format(
            txt_path,
            binary=False,
            unicode_errors="replace",
        )

Для того, чтобы понять какой препроцессинг требуется для данной модели, выведем первые 20 слов из её словаря. В контексте требуемой моделью предобработки можно отметить наличие POS-тегов и то, что все слова находятся в нижнем регистре.

In [19]:
list(model.key_to_index.keys())[:20]

['так_ADV',
 'быть_VERB',
 'мочь_VERB',
 'год_NOUN',
 'человек_NOUN',
 'xxxxxx_NUM',
 'сказать_VERB',
 'еще_ADV',
 'один_NUM',
 'говорить_VERB',
 'уже_ADV',
 'другой_ADJ',
 'время_NOUN',
 'xxxxxxxx_NUM',
 'знать_VERB',
 'сам_ADJ',
 'самый_ADJ',
 'делать_VERB',
 'дело_NOUN',
 'день_NOUN']

Далее возьмем 20 слов (по 5 на каждый домен) и посмотрим 10 ближайших соседей для каждого из них.

Ближайшими соседями некоторых слов являются их производные (например, котенок и котенка). Это говорит о том, что препроцессинг корпуса не предполагает лемматизацию.

Также можно отметить странность в тегах для некоторых слов: например, для слова "волчий" стоит тег "NOUN" (существительное), когда логичнее было бы использовать тег "ADJ" (прилагательное).

In [20]:
def print_neighbors(wv, word, top_n=10):
    if wv is None:
        print("Нет загруженных эмбеддингов (wv = None)")

        return

    if word not in wv:
        print(f"\"{word}\" нет в словаре модели")

        return

    neighbors = wv.most_similar(word, topn=top_n)
    print(word, "->", [w for w, _ in neighbors])


words = {
    "животные": ["кошка", "собака", "лошадь", "медведь", "волк"],
    "еда": ["хлеб", "молоко", "мясо", "сахар", "масло"],
    "семья": ["мать", "отец", "брат", "сестра", "дочь"],
    "погода": ["дождь", "снег", "ветер", "солнце", "мороз"],
}

for domain in words:
    print("Домен:", domain)

    for word in words[domain]:
        print_neighbors(model, word + "_NOUN")

Домен: животные
кошка_NOUN -> ['кот_NOUN', 'котенок_NOUN', 'котенка_NOUN', 'котенок_VERB', 'собака_NOUN', 'щенок_NOUN', 'собачка_NOUN', 'кошка_PROPN', 'дворняжка_NOUN', 'крыса_NOUN']
собака_NOUN -> ['пес_NOUN', 'дворняжка_NOUN', 'дворняга_NOUN', 'легавый_ADJ', 'кошка_NOUN', 'собачонка_NOUN', 'овчарка_NOUN', 'щенок_NOUN', 'гончий_ADJ', 'собачка_NOUN']
лошадь_NOUN -> ['конь_NOUN', 'лошадь_PROPN', 'жеребец_NOUN', 'лошадь_ADV', 'кобыла_NOUN', 'повозка_NOUN', 'лошадка_NOUN', 'телега_NOUN', 'иноходец_NOUN', 'мул_NOUN']
медведь_NOUN -> ['медведь_PROPN', 'медведь_VERB', 'зверь_NOUN', 'волк_NOUN', 'медведица_NOUN', 'барсук_NOUN', 'тигр_NOUN', 'заяц_NOUN', 'кабан_NOUN', 'медвежонок_NOUN']
волк_NOUN -> ['медведь_NOUN', 'зверь_NOUN', 'волк_PROPN', 'заяц_NOUN', 'волчица_NOUN', 'лисица_NOUN', 'лиса_NOUN', 'волчий_NOUN', 'койот_NOUN', 'волчонок_NOUN']
Домен: еда
хлеб_NOUN -> ['хлеба_NOUN', 'хлебец_NOUN', 'хлебушек_NOUN', 'хлебный_ADJ', 'хлеб_PROPN', 'хлебль_NOUN', 'пшено_NOUN', 'хлбъ_PROPN', 'мучица_

## Обучение Word2Vec и сравнение ближайших соседей с готовыми эмбеддингами

### Исходный датасет

В качестве исходного датасета используется [Kostya165/ru_emotion_dvach](https://huggingface.co/datasets/Kostya165/ru_emotion_dvach) для задач классификации эмоционального интента.

In [21]:
ru_emotion_dvach_ds = load_dataset("Kostya165/ru_emotion_dvach")
texts = ru_emotion_dvach_ds['train']['text']
labels = ru_emotion_dvach_ds['train']['label']

print("Количество документов:", len(texts))
print("Количество категорий", len(Counter(labels)))

README.md: 0.00B [00:00, ?B/s]

train_part_1.csv: 0.00B [00:00, ?B/s]

train_part_2.csv: 0.00B [00:00, ?B/s]

train_part_3.csv: 0.00B [00:00, ?B/s]

train_part_4.csv: 0.00B [00:00, ?B/s]

...

train_part_6.csv: 0.00B [00:00, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/59061 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2507 [00:00<?, ? examples/s]

Количество документов: 59061
Количество категорий 5


Сделаем простую токенизацию исходного корпуса.

In [26]:
def get_tokens(text: str) -> List[str]:
    text = text.lower()
    tokens = re.findall(r"[а-яё]+(?:-[а-яё]+)?", text)

    return tokens


texts_tokenized = [get_tokens(t) for t in texts if t]

print("Примеры токенов:", texts_tokenized[0][:30])

Примеры токенов: ['я', 'боюсь', 'что', 'из-за', 'контроля', 'сварных', 'соединений', 'я', 'могу', 'пропустить', 'что-то', 'важное', 'и', 'тогда', 'будут', 'большие', 'проблемы']


### Обучение Word2Vec

Обучим Word2Vec на полученных токенах.

In [27]:
w2v = Word2Vec(
    sentences=texts_tokenized,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,
    negative=10,
    epochs=10,
    workers=4,
)

wv = w2v.wv

print("Словарь:", len(wv))
print("Размерность:", wv.vector_size)

Словарь: 39092
Размерность: 100


Сравним ближайшие слова из обученного W2V с готовой моделью. Из полученных результатов можно заметить:

1. В список ближайших слов к слову "волк" есть прилагательное "тамбовский" (фразеологизм "тамбовский волк");
2. Рядом со словом "хлеб" присутствуют два слова – "литл" и "биг". Предположительно, это связанно с тем, что "хлеб" и "литл биг" – музыкальные группы и они обсуждаются вместе;
3. Соседнее со словом "мороз" – "снегурочка-дед" (Дед Мороз и Снегурочка).

In [29]:
for domain in words:
    print("Домен:", domain)

    for word in words[domain]:
        print_neighbors(model, word + "_NOUN")
        print_neighbors(wv, word)

    print()

Домен: животные
кошка_NOUN -> ['кот_NOUN', 'котенок_NOUN', 'котенка_NOUN', 'котенок_VERB', 'собака_NOUN', 'щенок_NOUN', 'собачка_NOUN', 'кошка_PROPN', 'дворняжка_NOUN', 'крыса_NOUN']
кошка -> ['клёвая', 'милая', 'бедная', 'сахарная', 'крупная', 'гуляла', 'умерла', 'лодочка', 'мелодия', 'тарелка']
собака_NOUN -> ['пес_NOUN', 'дворняжка_NOUN', 'дворняга_NOUN', 'легавый_ADJ', 'кошка_NOUN', 'собачонка_NOUN', 'овчарка_NOUN', 'щенок_NOUN', 'гончий_ADJ', 'собачка_NOUN']
собака -> ['кусала', 'стайная', 'роялю', 'поворачиваюсь', 'подумает', 'болен', 'встретит', 'летает', 'жадный', 'крик']
лошадь_NOUN -> ['конь_NOUN', 'лошадь_PROPN', 'жеребец_NOUN', 'лошадь_ADV', 'кобыла_NOUN', 'повозка_NOUN', 'лошадка_NOUN', 'телега_NOUN', 'иноходец_NOUN', 'мул_NOUN']
лошадь -> ['разбегу', 'стожок', 'брэда', 'учебным', 'питта', 'световым', 'пащенко', 'слепая', 'чирик', 'лгуньей']
медведь_NOUN -> ['медведь_PROPN', 'медведь_VERB', 'зверь_NOUN', 'волк_NOUN', 'медведица_NOUN', 'барсук_NOUN', 'тигр_NOUN', 'заяц_NOUN